In [2]:
#To deal with proxy data
import pandas as pd
import numpy as np
import json
import requests
import pandas as pd
import io
import ast
from pathlib import Path

In [3]:
url = 'https://linkedearth.graphdb.mint.isi.edu/repositories/LiPDVerse-dynamic'

query = """PREFIX le: <http://linked.earth/ontology#>
PREFIX wgs84: <http://www.w3.org/2003/01/geo/wgs84_pos#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT distinct?varID ?dataSetName ?lat ?lon ?varname ?interpLabel ?val ?varunits ?timevarname ?timeval ?timeunits ?archiveType where{

    ?ds a le:Dataset .
    ?ds le:hasName ?dataSetName .
    OPTIONAL{?ds le:hasArchiveType ?archiveTypeObj .
             ?archiveTypeObj rdfs:label ?archiveType .}
    
    
    ?ds le:hasLocation ?loc .
    ?loc wgs84:lat ?lat .
    FILTER(?lat<26 && ?lat>-26) 
    ?loc wgs84:long ?lon .
    FILTER(?lon<-70 && ?lon>-150) 
    
    ?ds le:hasPaleoData ?data .
    ?data le:hasMeasurementTable ?table .
    ?table le:hasVariable ?var .
    ?var le:hasName ?varname .
    VALUES ?varname {"d18O"} .
    ?var le:partOfCompilation  ?comp .
    ?comp le:hasName ?compName .
    VALUES ?compName {"iso2k" "Pages2kTemperature" "CoralHydro2k" "SISAL-LiPD"} .
    ?var le:hasInterpretation ?interp .
    ?interp le:hasVariable ?interpVar .
    ?interpVar rdfs:label ?interpLabel .
    FILTER (REGEX(?interpLabel, "precipitation.*", "i"))
    ?var le:hasVariableId ?varID .
    ?var le:hasValues ?val .
    OPTIONAL{?var le:hasUnits ?varunitsObj .
    		?varunitsObj rdfs:label ?varunits .}
    
    ?table le:hasVariable ?timevar .
    ?timevar le:hasName ?timevarname .
    VALUES ?timevarname {"year" "age"} .
    ?timevar le:hasValues ?timeval .
    OPTIONAL{?timevar le:hasUnits ?timeunitsObj .
    		 ?timeunitsObj rdfs:label ?timeunits .}  
}"""

In [4]:
response = requests.post(url, data = {'query': query})

data = io.StringIO(response.text)
df_res = pd.read_csv(data, sep=",")

df_res['val']=df_res['val'].apply(lambda row : json.loads(row) if isinstance(row, str) else row)
df_res['timeval']=df_res['timeval'].apply(lambda row : json.loads(row) if isinstance(row, str) else row)

df_res.head()

,varID,dataSetName,lat,lon,varname,interpLabel,val,varunits,timevarname,timeval,timeunits,archiveType
0,TR04EVLI01,TR04EVLI,10.0,-85.0,d18O,precipitationIsotope,"[23.69, 24.29, 24.25, 24.74, 25.7, 26.33, 26.0...",permil,year,"[2000.75, 2000.73, 2000.72, 2000.71, 2000.7, 2...",yr AD,Wood
1,TR04EVLI01,TR04EVLI,10.0,-85.0,d18O,precipitationIsotope,"[23.69, 24.29, 24.25, 24.74, 25.7, 26.33, 26.0...",permil,year,"[2000.75, 2000.73, 2000.72, 2000.71, 2000.7, 2...",yr AD,Wood
2,TR04EVLI01,TR04EVLI,10.0,-85.0,d18O,precipitationIsotope,"[23.69, 24.29, 24.25, 24.74, 25.7, 26.33, 26.0...",permil,year,"[2000.75, 2000.73, 2000.72, 2000.71, 2000.7, 2...",yr AD,Wood
3,TR04EVLI01,TR04EVLI,10.0,-85.0,d18O,precipitationIsotope,"[23.69, 24.29, 24.25, 24.74, 25.7, 26.33, 26.0...",permil,year,"[2000.75, 2000.73, 2000.72, 2000.71, 2000.7, 2...",yr AD,Wood
4,TR04EVLI01,TR04EVLI,10.0,-85.0,d18O,precipitation,"[23.69, 24.29, 24.25, 24.74, 25.7, 26.33, 26.0...",permil,year,"[2000.75, 2000.73, 2000.72, 2000.71, 2000.7, 2...",yr AD,Wood


In [9]:
df_res['interpLabel'].unique()
df_filt = df_res[df_res['interpLabel']== 'precipitation']
df_filt.head()


,varID,dataSetName,lat,lon,varname,interpLabel,val,varunits,timevarname,timeval,timeunits,archiveType
4,TR04EVLI01,TR04EVLI,10.0,-85.0,d18O,precipitation,"[23.69, 24.29, 24.25, 24.74, 25.7, 26.33, 26.0...",permil,year,"[2000.75, 2000.73, 2000.72, 2000.71, 2000.7, 2...",yr AD,Wood
5,TR04EVLI01,TR04EVLI,10.0,-85.0,d18O,precipitation,"[23.69, 24.29, 24.25, 24.74, 25.7, 26.33, 26.0...",permil,year,"[2000.75, 2000.73, 2000.72, 2000.71, 2000.7, 2...",yr AD,Wood
6,TR04EVLI01,TR04EVLI,10.0,-85.0,d18O,precipitation,"[23.69, 24.29, 24.25, 24.74, 25.7, 26.33, 26.0...",permil,year,"[2000.75, 2000.73, 2000.72, 2000.71, 2000.7, 2...",yr AD,Wood
7,TR04EVLI01,TR04EVLI,10.0,-85.0,d18O,precipitation,"[23.69, 24.29, 24.25, 24.74, 25.7, 26.33, 26.0...",permil,year,"[2000.75, 2000.73, 2000.72, 2000.71, 2000.7, 2...",yr AD,Wood
8,TR04EVLI01,TR04EVLI,10.0,-85.0,d18O,precipitation,"[23.69, 24.29, 24.25, 24.74, 25.7, 26.33, 26.0...",permil,year,"[2000.75, 2000.73, 2000.72, 2000.71, 2000.7, 2...",yr AD,Wood
